In [ ]:
from pathlib import Path
import ixmp4
import pyam
import nomenclature

In [ ]:
platform = ixmp4.Platform("scenariocompass-transfer")

In [ ]:
df = pyam.concat(
    [
        i for i in list(Path("raw/NAVIGATE/").iterdir())
        if "" in str(i)
    ]
)

In [ ]:
df.rename(
    model={
        "GEM-E3_V2023": "GEM-E3 V2023",
    },
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (i, "NAVIGATE Demand-" + i[8:].replace("15C", "1.5°C").replace("20C", "2.0°C"))
             for i in df.scenario if i.startswith("NAV_Dem") 
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i, 
                (
                    ("NAVIGATE Supply-" + i[4:])
                    .replace("1p5C", "1.5°C")
                    .replace("2C", "2.0°C")
                    .replace("_", "-")
                    .replace("Default", "default")
                )
            ) for i in df.scenario if i.startswith("SUP")
        ]
    ),
    inplace=True,
)

In [ ]:
df.scenario

In [ ]:
df.model

In [ ]:
#definition = nomenclature.DataStructureDefinition("../definitions/")
definition = nomenclature.DataStructureDefinition("../../common-definitions/definitions/")

In [ ]:
df.region

In [ ]:
df.rename(
    region=dict(
        [(i, i.replace("GEM-E3_V2023", "GEM-E3 V2023")) for i in df.filter(region="GEM*").region]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    region={
        "MESSAGEix-GLOBIOM 1.1-R12|Rest Centrally Planned Asia": "MESSAGEix-GLOBIOM 1.1-R12|Rest of Centrally Planned Asia",
        "MESSAGEix-GLOBIOM 1.1-R12|Sub-saharan Africa": "MESSAGEix-GLOBIOM 1.1-R12|Sub-Saharan Africa",
        "European Union (28 member countries)": "European Union and United Kingdom",
        "Russia": "Russian Federation",
        "IMAGE 3.3|C. Europe": "IMAGE 3.3|Central Europe",
        "COFFEE 1.5|European Union": "COFFEE 1.5|Europe",
        "IMAGE 3.3|China": "IMAGE 3.3|China Region",
        "IMAGE 3.3|E. Africa": "IMAGE 3.3|Eastern Africa",
        "IMAGE 3.3|Indonesia": "IMAGE 3.3|Indonesia Region",
        "IMAGE 3.3|Kazakhstan region": "IMAGE 3.3|Central Asia",
        "IMAGE 3.3|Korea": "IMAGE 3.3|Korea Region",
        "IMAGE 3.3|N. Africa": "IMAGE 3.3|Northern Africa",
        "IMAGE 3.3|Rest C. America": "IMAGE 3.3|Central America",
        "IMAGE 3.3|Rest S. Africa": "IMAGE 3.3|Rest of Southern Africa",
        "IMAGE 3.3|Rest S. America": "IMAGE 3.3|Rest of South America",
        "IMAGE 3.3|Rest S. Asia": "IMAGE 3.3|Rest of South Asia",
        "IMAGE 3.3|Russia": "IMAGE 3.3|Russia Region",
        "IMAGE 3.3|SE. Asia": "IMAGE 3.3|Southeastern Asia",
        "IMAGE 3.3|USA": "IMAGE 3.3|United States",
        "IMAGE 3.3|Ukraine region": "IMAGE 3.3|Ukraine Region",
        "IMAGE 3.3|W. Africa": "IMAGE 3.3|Western Africa",
        "IMAGE 3.3|W. Europe": "IMAGE 3.3|Western Europe",
        "REMIND 3.0|China": "REMIND 3.0|China and Taiwan",
        "REMIND 3.0|Countries from the Reforming Ecomonies of the Former Soviet Union": "REMIND 3.0|Russia and Reforming Economies",
        "REMIND 3.0|Middle East, North Africa, Central Asia": "REMIND 3.0|Middle East and North Africa",
        "WITCH 5.0|Japan and South Korea": "WITCH 5.0|Japan and Korea",
        "WITCH 5.0|Latin america and Caraibes (except Brasil and Mexico)": "WITCH 5.0|Latin America and the Caribbean",  
        "WITCH 5.0|Moyen-Orient and North Africa": "WITCH 5.0|Middle East and North Africa", 
        "WITCH 5.0|Oceania": "WITCH 5.0|Australia, New Zealand, and Oceania islands", 
        "WITCH 5.0|South-East Asia": "WITCH 5.0|South East Asia", 
        "WITCH 5.0|Sub-Saharian African (except South Africa)": "WITCH 5.0|Sub-Saharan Africa", 
        "WITCH 5.0|Transition Economies (including Russia)": "WITCH 5.0|Non-EU Eastern European and Transition Countries", 
        "WITCH 5.0|United States of America": "WITCH 5.0|United States", 
        "JRC-GEM-E3 v2021|Rest of Euroasia": "JRC-GEM-E3 v2021|Rest of Eurasia",
        "GEM-E3 V2023|Rest of fossil fuel producers": "GEM-E3 V2023|Rest of Fossil Fuel Producers",
        "GEM-E3 V2023|Rest of the world": "GEM-E3 V2023|Rest of the World",
    },
    inplace=True,
)

In [ ]:
df.region

In [ ]:
df.filter(model="JRC-GEM-E3 v2021", keep=False, inplace=True)

In [ ]:
[i for i in df.variable if "Cement" in i]

In [ ]:
ex_post_addition = {
    "Production|Iron and Steel|Volume": "Production|Iron and Steel|Steel",
    "Value Added|Industry|Iron and Steel": "Value Added|Industry|Iron and Steel",
    "Emissions|CO2|Energy|Demand|Industry|Steel": "Emissions|CO2|Energy|Demand|Industry|Iron and Steel",
    "Emissions|CO2|Industrial Processes|Non-Metallic Minerals|Cement": "Emissions|CO2|Industrial Processes|Cement",
}

In [ ]:
df = df.filter(variable=ex_post_addition.keys()).rename(variable=ex_post_addition)          

In [ ]:
definition.validate(df, dimensions=["region"])

In [ ]:
project = ["navigate", "engage"]
legacy_mapping = {}

for _project in project:
    for code, attrs in definition.variable.items():
        if _project in attrs.extra_attributes:
            legacy_mapping[attrs.__getattr__(_project)] = code
    
    df.rename(variable=legacy_mapping, inplace=True)

In [ ]:
# rename units
df.rename(
    unit={
        "US$2010/kW OR local currency/kW": "USD_2010/kW",
        "US$2010/kW": "USD_2010/kW",
        "billion US$2010/yr": "billion USD_2010/yr",
        "billion US$2010/yr OR local currency/yr": "billion USD_2010/yr",
        "billion US$2010/yr or local currency/yr": "billion USD_2010/yr",
        "US$2010/t CO2": "USD_2010/t CO2",
        "US$2010/tCO2": "USD_2010/t CO2",
        "US$2010/t CO2 or local currency/t CO2": "USD_2010/t CO2",
        "million Ha/yr": "million ha",
        "Million": "million",
        "Mt NOx/yr": "Mt NO2/yr",  
        "Mt N2O/yr": "kt N2O/yr",
        "US$2010/GJ": "USD_2010/GJ",
        "bn m2": "billion m2",
        "bn tkm/yr": "billion tkm/yr",
        "bn pkm/yr": "billion pkm/yr",
    },
    inplace=True,
)

In [ ]:
# update carbon-management variables
variable_mapping = {
    "Agricultural Demand|Crops|Energy": "Agricultural Demand|Crops|Bioenergy",
    "Agricultural Demand|Crops|Energy|1st generation": "Agricultural Demand|Crops|Bioenergy|1st Generation",
    "Agricultural Demand|Crops|Energy|2nd generation": "Agricultural Demand|Crops|Bioenergy|2nd Generation",
    "Carbon Sequestration|CCS": "Carbon Capture|Geological Storage",
    "Carbon Sequestration|CCS|Biomass": "Carbon Capture|Geological Storage|Biomass",
    "Carbon Sequestration|CCS|Biomass|Energy|Supply": "Carbon Capture|Energy|Supply|Biomass",
    "Carbon Sequestration|CCS|Fossil": "Carbon Capture|Energy|Fossil",
    "Carbon Sequestration|CCS|Fossil|Energy|Demand|Industry": "Carbon Capture|Energy|Demand|Industry",
    "Carbon Sequestration|CCS|Fossil|Energy|Supply": "Carbon Capture|Energy|Supply|Fossil",
    "Carbon Sequestration|CCS|Industrial Processes": "Carbon Capture|Industrial Processes",
    "Carbon Sequestration|Land Use|Afforestation": "Carbon Removal|Land Use|Re/Afforestation",
    "Carbon Sequestration|Direct Air Capture": "Carbon Removal|Geological Storage|Direct Air Capture",
    "Carbon Sequestration|Enhanced Weathering": "Carbon Removal|Enhanced Weathering",
    "Carbon Sequestration|Land Use": "Carbon Removal|Land Use",
    "Yield|Cereal": "Yield|Cropland|Cereals",
    "Yield|Oilcrops": "Yield|Cropland|Oil Crops",
    "Yield|Sugarcrops": "Yield|Cropland|Sugar Crops",
}

df.rename(variable=variable_mapping, inplace=True)

In [ ]:
# remove variables of little relevance that are not included in common-definitions
df.filter(
    variable=[
        "*AR6 climate diagnostics*",
        "Capacity Additions|Electricity|Storage Capacity",
        "Capacity|Electricity|Storage",
        "Cumulative Capacity*",
        "Secondary Energy",
        "Diagnostics|MAGICC6*",
        "Forcing*",
        "OM Cost*"
    ],
    keep=False,
    inplace=True
)

In [ ]:
definition.validate(df, dimensions=["variable"])

In [ ]:
df.filter(variable="Labor Force|Employed|*", keep=False, inplace=True)

In [ ]:
validation_args = ["upper_bound", "lower_bound", "value", "rtol", "atol", "range"]

validation_list = list()

for name, variable in definition.variable.items():
    if any([i in validation_args for i in variable.extra_attributes]):
        validation_list.append(
            dict(
                variable=name,
                validation=[dict([(key, value) for key, value in variable.extra_attributes.items() if key in validation_args])]
            )
        )

In [ ]:
validator = nomenclature.processor.DataValidator(criteria_items=validation_list, file=".")

In [ ]:
corrected_df = df.filter(model="COFFEE 1.5", variable="Carbon Capture|Industrial Processes")
corrected_df._data = - corrected_df._data

df = pyam.concat(
    [
        df.filter(model="COFFEE 1.5", variable="Carbon Capture|Industrial Processes", keep=False),
        corrected_df,
    ]
)

In [ ]:
# incorrectly aggregated variables
df.filter(
    variable=[
        "Terrestrial Biodiversity|Biodiversity Intactness Index",
        "Terrestrial Biodiversity|Mean Species Abundance|Plants",
        "Consumption|*",
        "Income|*",
        "Efficiency|*",
    ],
    region=[
        "World",
        "*(R5)",
        "*(R9)",
        "*(R10)",
    ],
    keep=False,
    inplace=True,
)

df.filter(
    model="IMAGE 3.3",
    variable=[
        "Terrestrial Biodiversity|Biodiversity Intactness Index",
    ],
    region=[
        "European Union and United Kingdom",
    ],
    keep=False,
    inplace=True,
)

df.filter(
    model="MESSAGEix-GLOBIOM 1.1-BMT-R12",
    variable=[
        "Efficiency*",
    ],
    region=[
        "European Union and United Kingdom",
    ],
    keep=False,
    inplace=True,
)

In [ ]:
validation_args

In [ ]:
# reporting errors
df.filter(model="COFFEE 1.5", variable="Carbon Capture|Industrial Processes", keep=False, inplace=True)

In [ ]:
df.model

In [ ]:
validator.apply(df)

In [ ]:
definition.validate(df)

In [ ]:
df.set_meta("NAVIGATE [Horizon 2020]", "Project")
df.set_meta("van Heerden et al. (2025)", "Scientific Manuscript (Citation)")
df.set_meta("10.1038/s41560-025-01703-1", "Scientific Manuscript (DOI)")
#df.set_meta("10.5281/zenodo.13989530", "Data Source (DOI)")

In [ ]:
df_upload = df.filter(variable=definition.variable)
df_upload

In [ ]:
platform = ixmp4.Platform("scenariocompass-transfer")

In [ ]:
for model in df_upload.model:
    df_upload.filter(model=model).to_ixmp4(platform)
    print(model)

In [ ]:
for model, scenario in df.index:
    run = platform.runs.get(model, scenario)
    _df = df.filter(model=model, scenario=scenario)
    with run.transact("Add variables for UNEP-FI"):
        run.iamc.add(_df.data)
        print(f"Done {model} | {scenario}")